In [1]:
import os
import numpy as np
import pandas as pd
import shutil
import librosa
from tqdm import tqdm

In [2]:
from datasets import load_dataset

/media/muzaffar/Data/projects/nlp_audio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# datasets >= 4.0 da loading script'lar (common_voice_13_0.py) olib tashlangan,
# shuning uchun mozilla-foundation/common_voice_13_0 ni to'g'ridan-to'g'ri yuklab bo'lmaydi.
# Uning o'rniga parquet ko'rinishidagi mirror'dan foydalanamiz (Common Voice 17.0, uzbek).
dataset = load_dataset("yakhyo/mozilla-common-voice-uzbek", split="train+validation")
dataset

Dataset({
    features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant', 'text'],
    num_rows: 60609
})

In [4]:
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [6]:
# Audio'ni 16 kHz ga keltiramiz (feature extraction uchun qulay)
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# datasets >= 4.0 da audio ustuni dict emas, AudioDecoder obyekti qaytaradi.
# numpy array olish uchun get_all_samples() ishlatiladi.
def to_array(audio):
    samples = audio.get_all_samples()
    return samples.data.numpy().squeeze(), samples.sample_rate

ex = dataset[0]
y, sr = to_array(ex["audio"])
print("shape:", y.shape, "| sr:", sr)
print("gender:", ex["gender"], "| age:", ex["age"])
print("sentence:", ex["sentence"])

shape: (76608,) | sr: 16000
gender: male_masculine | age: twenties
sentence: Bugun ertalab Gyotenikiga taklifnoma oldim.


In [7]:
from collections import Counter

# Dataset'ning ~44% qismida gender belgisi yo'q (bo'sh satr) - ularni tashlaymiz.
print("filtrlashdan oldin:", Counter(dataset["gender"]))

LABELS = {"male_masculine": 0, "female_feminine": 1}

labeled = dataset.filter(lambda x: x["gender"] in LABELS)
labeled = labeled.map(lambda x: {"label": LABELS[x["gender"]]})

print("filtrlashdan keyin:", Counter(labeled["gender"]))
print("jami:", len(labeled))

filtrlashdan oldin: Counter({'': 26956, 'male_masculine': 21412, 'female_feminine': 12241})
filtrlashdan keyin: Counter({'male_masculine': 21412, 'female_feminine': 12241})
jami: 33653


In [8]:
# Metadata'ni pandas'ga o'tkazamiz. audio ustunini tashlaymiz, chunki unda
# har bir yozuvning mp3 baytlari bor va to'liq DataFrame RAM'ni bekorga to'ldiradi.
meta_df = labeled.remove_columns(["audio"]).to_pandas()
meta_df.head()

,client_id,sentence,up_votes,down_votes,age,gender,accent,locale,segment,variant,text,label
0,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Bugun ertalab Gyotenikiga taklifnoma oldim.,2,0,twenties,male_masculine,,uz,,,Bugun ertalab Gyotenikiga taklifnoma oldim.,0
1,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Uning badiiy tasvir imkoniyatlarini rivojlanti...,2,0,twenties,male_masculine,,uz,,,Uning badiiy tasvir imkoniyatlarini rivojlanti...,0
2,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Udan ko’ra balandroq joy bor.,2,1,twenties,male_masculine,,uz,,,Udan ko'ra balandroq joy bor.,0
3,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Bu jumlada fig‘oni falakka chiqib birikmasi ib...,2,1,twenties,male_masculine,,uz,,,Bu jumlada fig'oni falakka chiqib birikmasi ib...,0
4,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Bundan tashqari puxta jamlangan kutubxona bor.,2,1,twenties,male_masculine,,uz,,,Bundan tashqari puxta jamlangan kutubxona bor.,0


In [9]:
import pandas as pd

In [ ]:
N_MFCC = 40

def extract_features(batch):
    vectors = []
    for audio in batch["audio"]:
        samples = audio.get_all_samples()
        y = samples.data.numpy().squeeze()
        mfcc = librosa.feature.mfcc(y=y, sr=samples.sample_rate, n_mfcc=N_MFCC)
        vectors.append(mfcc.mean(axis=1))
    return {"features": vectors}

In [11]:
# ~7 daqiqa (num_proc bilan tezroq). Natija diskka cache'lanadi,
# ikkinchi marta ishga tushirilsa qayta hisoblanmaydi.
featured = labeled.map(
    extract_features,
    batched=True,
    batch_size=64,
    remove_columns=["audio"],
    num_proc=4,
)

X = np.array(featured["features"])
y = np.array(featured["label"])

print("X:", X.shape, "| y:", y.shape)
print("klasslar:", np.bincount(y), "(0=erkak, 1=ayol)")

X: (33653, 40) | y: (33653,)
klasslar: [21412 12241] (0=erkak, 1=ayol)


In [16]:
new_df = meta_df[["gender"]].copy()
new_df.insert(0, "idx", range(len(new_df)))
new_df["label"] = meta_df["label"]

new_df = new_df[new_df["gender"].isin(["male_masculine", "female_feminine"])]

print(new_df.shape)
print(new_df["gender"].value_counts())
new_df.head()

(33653, 3)
gender
male_masculine     21412
female_feminine    12241
Name: count, dtype: int64


,idx,gender,label
0,0,male_masculine,0
1,1,male_masculine,0
2,2,male_masculine,0
3,3,male_masculine,0
4,4,male_masculine,0


In [19]:
male = new_df[new_df["gender"] == "male_masculine"]
female = new_df[new_df["gender"] == "female_feminine"]
print(male.shape, female.shape)

(21412, 3) (12241, 3)


In [21]:
RANDOM_STATE = 42

male   = new_df[new_df["gender"] == "male_masculine"]
female = new_df[new_df["gender"] == "female_feminine"]
n = min(len(male), len(female))

balanced_df = pd.concat([
    male.sample(n=n, random_state=RANDOM_STATE),
    female.sample(n=n, random_state=RANDOM_STATE),
])
# aralashtiramiz, aks holda avval hamma erkak, keyin hamma ayol bo'lib qoladi
balanced_df = balanced_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# idx orqali feature'larni ham xuddi shu tartibda tanlaymiz
X_bal = X[balanced_df["idx"].to_numpy()]
y_bal = balanced_df["label"].to_numpy()

print("balanced_df:", balanced_df.shape)
print(balanced_df["gender"].value_counts().to_string())
print("X_bal:", X_bal.shape, "| klasslar:", np.bincount(y_bal))

balanced_df: (24482, 3)
gender
male_masculine     12241
female_feminine    12241
X_bal: (24482, 40) | klasslar: [12241 12241]


In [22]:
balanced_df.iloc[0].path

AttributeError: 'Series' object has no attribute 'path'